In [ ]:
 # SETUP
from google.colab import drive
drive.mount('/content/drive')

import os
import mne
import numpy as np
import matplotlib.pyplot as plt

## Time-Frequency Analysis (Morlet Wavelets)


In [ ]:
# === Configuration ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
freqs = np.arange(4, 40, 1)
n_cycles = freqs / 2
baseline = (-0.1, 0.0)
decim = 3
reflect_ms = 0.1  # 100 ms reflection

# === Predefined Brain Regions ===
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

# === Initialize storage ===
all_subject_spectra = {}

# === Loop through all participants ===
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print(f"\n📊 Power Spectrum for participant {subj}")

    clean_epo_path = os.path.join(root_dir, subj, f"{subj}_O1_epo_clean.fif")
    if not os.path.exists(clean_epo_path):
        print(f"❌ Missing data for {subj}, skipping.")
        continue

    # === Load epochs ===
    epochs = mne.read_epochs(clean_epo_path, preload=True)
    sfreq = epochs.info['sfreq']
    reflect_samples = int(reflect_ms * sfreq)

    # === Reflect the data ===
    data = epochs.get_data()  # (n_epochs, n_channels, n_times)
    reflected_data = np.concatenate([
        data[:, :, reflect_samples:0:-1],
        data,
        data[:, :, -2:-reflect_samples-2:-1]
    ], axis=2)

    # Create new time vector for reflected data
    time_step = 1.0 / sfreq
    extra_time_left = epochs.times[0] - np.arange(reflect_samples, 0, -1) * time_step
    extra_time_right = epochs.times[-1] + np.arange(1, reflect_samples + 1) * time_step
    reflected_times = np.concatenate([extra_time_left, epochs.times, extra_time_right])

    # === Compute TFR manually ===
    picks = mne.pick_types(epochs.info, eeg=True, exclude='bads')
    power = mne.time_frequency.tfr_array_morlet(
        reflected_data[:, picks],
        sfreq=sfreq,
        freqs=freqs,
        n_cycles=n_cycles,
        output='power'
    )

    # Clip reflection edges
    clip_samples = reflect_samples
    power = power[:, :, :, clip_samples:-clip_samples]
    power_times = reflected_times[clip_samples:-clip_samples]

    # Apply baseline correction
    baseline_mask = (power_times >= baseline[0]) & (power_times <= baseline[1])
    baseline_vals = power[:, :, :, baseline_mask].mean(axis=-1, keepdims=True)
    power = np.log10(power / baseline_vals)

    # === Plot region spectra ===
    plt.figure(figsize=(10, 6))
    subject_spectra = {}

    for region, chan_list in regions.items():
        region_chs = [ch for ch in chan_list if ch in epochs.ch_names]
        if not region_chs:
            continue

        ch_idxs = [epochs.ch_names.index(ch) for ch in region_chs]
        region_data = power[:, ch_idxs, :, :].mean(axis=1)  # (epochs, freqs, times)
        power_spectrum = region_data.mean(axis=0).mean(axis=1)

        subject_spectra[region] = power_spectrum
        plt.plot(freqs, power_spectrum, label=region)

    all_subject_spectra[subj] = subject_spectra

    plt.title(f'Participant {subj} — Region-wise Power Spectrum')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Power (log-ratio)')
    plt.legend(loc='upper right', fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## Calculate average theta and beta power features for each region

In [ ]:

# Example inputs
freqs = np.arange(4, 40, 1)        # Frequency array from your TFD
times = np.linspace(-0.1, 0.8, 226) # Time array (adjust based on your epochs)
regions_of_interest = ['Left Anterior', 'Medial Anterior', 'Right Anterior',
                       'Left Central', 'Medial Central', 'Right Central']

# Time windows in seconds
n1_window = (0.1, 0.15)
n280_window = (0.2, 0.3)

# Frequency bands
theta_band = (4, 7)
beta_band = (13, 20)

participant_results = []

for subj, spectra in all_subject_spectra.items():  # from your notebook
    subj_result = {'Participant': subj}

    for region in regions_of_interest:
        region_power = spectra[region]  # (freqs) averaged over time in your code

        # Extract indices for theta and beta
        theta_idx = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
        beta_idx = (freqs >= beta_band[0]) & (freqs <= beta_band[1])

        # Since your current power is frequency-only averaged, we need time-locked power.
        # If you have full TFR [freq x time], you would subset by time index here.
        # Example: theta_power = power[theta_idx, n1_time_idx].mean()

        subj_result[f'{region}_Theta'] = region_power[theta_idx].mean()
        subj_result[f'{region}_Beta'] = region_power[beta_idx].mean()

    participant_results.append(subj_result)

df_results = pd.DataFrame(participant_results)

from IPython.display import display
display(df_results)



## Participant-level summary table of theta (N1) and beta (N280) power across all brain regions

In [ ]:
# Extract theta and beta columns
theta_cols = [col for col in df_results.columns if '_Theta' in col]
beta_cols = [col for col in df_results.columns if '_Beta' in col]

# Create summary DataFrame
df_summary = pd.DataFrame({
    'Participant': df_results['Participant'],
    'Theta_Power_N1': df_results[theta_cols].mean(axis=1),
    'Beta_Power_N280': df_results[beta_cols].mean(axis=1)
})

from IPython.display import display
display(df_summary)

# Save as a new separate file
save_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_Spectral_Only.csv"
df_summary.to_csv(save_path, index=False)

print(f"Spectral summary saved successfully to: {save_path}")



Participants 8, 14, 15, 26: Higher theta and/or mild beta power, likely stronger early word perception.

Participants 13, 17, 19, 20: Negative theta and beta power, suggesting weaker perception of function words (possibly more influenced by slowed context).

Most other participants fall in between, showing mixed perceptual engagement.



## Load Behavioral Data

In [ ]:
import pandas as pd

# Load the CSV
file_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_includeNA.csv"
df = pd.read_csv(file_path)

# Work on a copy so the original file is untouched
df_behavior = df.copy()


## Convert to Perception Rate
Since 0.0 means function word perceived, we invert it so that 1 = perceived:

In [ ]:

# Convert Correct_Sequence to perception (1 = perceived, 0 = not perceived)
df_behavior['Perceived'] = df_behavior['Correct_Sequence (0/1)'].apply(lambda x: 1 if x == 0 else 0)

# Show the first few rows
print(df_behavior[['Participant', 'Correct_Sequence (0/1)', 'Perceived']].head())


## Compute mean perception rate per participant

In [ ]:
df_behavior_summary = df_behavior.groupby('Participant')['Perceived'].mean().reset_index()
df_behavior_summary.rename(columns={'Perceived': 'Perception_Rate'}, inplace=True)

print(df_behavior_summary)


## Merging EEG spectral features (theta and beta power) with behavioral performance data (perception rate) for each participan

In [ ]:
import pandas as pd

# Load spectral summary
spectral_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_Spectral_Only.csv"
df_spectral = pd.read_csv(spectral_path)  # This comes from df_summary

# Load behavioral data
behavior_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_includeNA.csv"
df_behavior = pd.read_csv(behavior_path)

# Copy and transform behavior
df_behavior_copy = df_behavior.copy()
df_behavior_copy['Perceived'] = df_behavior_copy['Correct_Sequence (0/1)'].apply(lambda x: 1 if x == 0 else 0)

# Average perception per participant
df_behavior_summary = df_behavior_copy.groupby('Participant')['Perceived'].mean().reset_index()
df_behavior_summary.rename(columns={'Perceived': 'Perception_Rate'}, inplace=True)

# Merge spectral and behavioral summaries
df_merged = pd.merge(df_spectral, df_behavior_summary, on="Participant", how="outer")
print(df_merged)



### Key Observations

**High Theta/Beta Power & Higher Perception Rate:**
- P8: Θ = 0.0878, β = 0.0136, Perception = 0.1705
- P14: Θ = 0.0667, β = -0.0214, Perception = 0.1860
- P26: Θ = 0.0385, β = -0.0180, Perception = 0.1550  

These participants show relatively higher spectral power and higher perception rates, consistent with stronger function word processing.

**Negative Theta/Beta Power & Lower Perception Rate:**
- P13: Θ = 0.0255, β = -0.1177, Perception = 0.0930
- P17: Θ = -0.0654, β = -0.0491, Perception = 0.0698
- P19: Θ = -0.0222, β = -0.0335, Perception = 0.1473 (moderate)  

These participants show lower spectral power and weaker perception, aligning with ERP findings (reduced N1/N280).

**Missing Spectral Data:**  
Participants 2, 4, 5, 9, 16, and 23 have behavioral data only but no EEG spectral data (NaNs). This is expected because they were excluded from the spectral analysis (likely due to missing or noisy EEG data).
